# K2 Think v2 Raw API Tester

This notebook sends direct requests to the K2 Think v2 chat API using the same payload shape MindWeave uses in `backend/app/services/llm_gateway.py`.

It is set up to:
- load `.env` values from the repo root when possible
- call either the chat or agent endpoint
- print the exact JSON request payload
- show the untouched raw JSON response for each query
- optionally save batch results to disk for later inspection


In [ ]:
import json
import os
import time
from pathlib import Path

import httpx
from IPython.display import JSON, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "backend").exists() and (candidate / "README.md").exists():
            return candidate
    return current


def load_env_file(path: Path) -> dict[str, str]:
    loaded: dict[str, str] = {}
    if not path.exists():
        return loaded
    for line in path.read_text(encoding="utf-8").splitlines():
        raw = line.strip()
        if not raw or raw.startswith("#") or "=" not in raw:
            continue
        key, value = raw.split("=", 1)
        value = value.strip().strip('"').strip("'")
        loaded[key] = value
        os.environ.setdefault(key.strip(), value)
    return loaded


def mask_secret(value: str | None) -> str:
    if not value:
        return "<missing>"
    if len(value) <= 8:
        return "*" * len(value)
    return f"{value[:4]}...{value[-4:]}"


REPO_ROOT = find_repo_root()
ENV_PATH = REPO_ROOT / ".env"
LOADED_ENV = load_env_file(ENV_PATH)

K2_API_KEY = os.getenv("MW_K2_API_KEY") or os.getenv("K2_API_KEY")
K2_MODEL = os.getenv("MW_K2_MODEL", "MBZUAI-IFM/K2-Think-v2")
K2_CHAT_URL = os.getenv("MW_K2_CHAT_BASE_URL", "https://api.k2think.ai/v1/chat/completions")
K2_AGENT_URL = os.getenv("MW_K2_AGENT_BASE_URL", K2_CHAT_URL)
K2_REASONING_EFFORT = os.getenv("MW_K2_REASONING_EFFORT", "high")
K2_TEMPERATURE = float(os.getenv("MW_K2_TEMPERATURE", "0.8"))
K2_TOP_P = float(os.getenv("MW_K2_TOP_P", "1.0"))

if not K2_API_KEY:
    raise RuntimeError(
        "Missing K2 API key. Set MW_K2_API_KEY or K2_API_KEY in your shell or repo .env file before running this notebook."
    )

print(json.dumps({
    "repo_root": str(REPO_ROOT),
    "env_loaded_from": str(ENV_PATH) if ENV_PATH.exists() else "<not found>",
    "model": K2_MODEL,
    "chat_url": K2_CHAT_URL,
    "agent_url": K2_AGENT_URL,
    "reasoning_effort": K2_REASONING_EFFORT,
    "temperature": K2_TEMPERATURE,
    "top_p": K2_TOP_P,
    "api_key": mask_secret(K2_API_KEY),
}, indent=2))


In [ ]:
def build_messages(user_prompt: str, system_prompt: str | None = None, context: dict | None = None) -> list[dict[str, str]]:
    messages: list[dict[str, str]] = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    content = user_prompt
    if context:
        content = f"{user_prompt}\n\nContext:\n{json.dumps(context, indent=2)}"
    messages.append({"role": "user", "content": content})
    return messages


def build_k2_payload(
    user_prompt: str,
    system_prompt: str | None = None,
    context: dict | None = None,
    model: str = K2_MODEL,
    temperature: float = K2_TEMPERATURE,
    top_p: float = K2_TOP_P,
    reasoning_effort: str = K2_REASONING_EFFORT,
    seed: int | None = None,
    max_tokens: int | None = None,
) -> dict:
    payload = {
        "model": model,
        "messages": build_messages(user_prompt=user_prompt, system_prompt=system_prompt, context=context),
        "stream": False,
        "temperature": temperature,
        "top_p": top_p,
        "chat_template_kwargs": {"reasoning_effort": reasoning_effort},
    }
    if max_tokens is not None:
        payload["max_tokens"] = max_tokens
    if seed is not None:
        payload["seed"] = seed
    return payload


def call_k2(
    user_prompt: str,
    system_prompt: str | None = None,
    context: dict | None = None,
    *,
    agentic: bool = True,
    model: str = K2_MODEL,
    temperature: float = K2_TEMPERATURE,
    top_p: float = K2_TOP_P,
    reasoning_effort: str = K2_REASONING_EFFORT,
    seed: int | None = None,
    max_tokens: int | None = None,
    timeout: float = 90.0,
) -> dict:
    url = K2_AGENT_URL if agentic else K2_CHAT_URL
    payload = build_k2_payload(
        user_prompt=user_prompt,
        system_prompt=system_prompt,
        context=context,
        model=model,
        temperature=temperature,
        top_p=top_p,
        reasoning_effort=reasoning_effort,
        seed=seed,
        max_tokens=max_tokens,
    )
    with httpx.Client(timeout=timeout) as client:
        response = client.post(
            url,
            headers={
                "accept": "application/json",
                "Authorization": f"Bearer {K2_API_KEY}",
                "Content-Type": "application/json",
            },
            json=payload,
        )

    raw_text = response.text
    raw_json = None
    try:
        raw_json = response.json()
    except Exception:
        raw_json = None

    return {
        "ok": response.is_success,
        "status_code": response.status_code,
        "url": url,
        "request_payload": payload,
        "response_headers": dict(response.headers),
        "raw_text": raw_text,
        "raw_json": raw_json,
    }


def assistant_content(raw_json: dict | None) -> str | None:
    if not isinstance(raw_json, dict):
        return None
    try:
        return raw_json["choices"][0]["message"]["content"]
    except Exception:
        return None


def parse_json_string(value: str | None):
    if not isinstance(value, str):
        return None
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        decoder = json.JSONDecoder()
        extracted = None
        for index, char in enumerate(value):
            if char != "{":
                continue
            try:
                candidate, _ = decoder.raw_decode(value[index:])
            except json.JSONDecodeError:
                continue
            if isinstance(candidate, dict):
                extracted = candidate
        return extracted


def show_result(result: dict) -> None:
    print(f"POST {result['url']}")
    print(f"HTTP {result['status_code']} | ok={result['ok']}")
    print("\nRequest payload:")
    display(JSON(result["request_payload"]))

    if result["raw_json"] is not None:
        print("\nRaw response JSON:")
        display(JSON(result["raw_json"]))
    else:
        print("\nRaw response text:")
        print(result["raw_text"])

    content = assistant_content(result["raw_json"])
    if content:
        print("\nAssistant content:")
        print(content)
        parsed = parse_json_string(content)
        if parsed is not None:
            print("\nAssistant content parsed as JSON:")
            display(JSON(parsed))


def run_batch(
    prompts: list[str],
    system_prompt: str | None = None,
    *,
    agentic: bool = True,
    context_factory=None,
    temperature: float = K2_TEMPERATURE,
    top_p: float = K2_TOP_P,
    reasoning_effort: str = K2_REASONING_EFFORT,
    seed: int | None = None,
    max_tokens: int | None = None,
    pause_seconds: float = 0.0,
) -> list[dict]:
    results: list[dict] = []
    for index, prompt in enumerate(prompts, start=1):
        context = context_factory(prompt, index) if callable(context_factory) else context_factory
        result = call_k2(
            user_prompt=prompt,
            system_prompt=system_prompt,
            context=context,
            agentic=agentic,
            temperature=temperature,
            top_p=top_p,
            reasoning_effort=reasoning_effort,
            seed=seed,
            max_tokens=max_tokens,
        )
        result["query_index"] = index
        result["user_prompt"] = prompt
        results.append(result)

        print(f"\n=== Query {index} ===")
        print(prompt)
        show_result(result)

        if pause_seconds > 0:
            time.sleep(pause_seconds)
    return results


In [ ]:
single_result = call_k2(
    user_prompt="List three concrete reasons reasoning graphs can help enterprise audit workflows.",
    system_prompt="You are a precise assistant. Return a concise answer.",
    context={"runner": "notebook", "mode": "single_query_demo"},
    agentic=True,
    temperature=0.0,
    top_p=1.0,
    seed=42,
    max_tokens=800,
)

show_result(single_result)


In [ ]:
TEST_PROMPTS = [
    "Explain the difference between a reasoning model and a standard instruction-tuned chat model in two short paragraphs.",
    "Return a JSON object with keys summary and use_cases describing when graph-based reasoning is useful.",
    "Give five questions I should ask when evaluating an agentic reasoning API for production use.",
]

batch_results = run_batch(
    TEST_PROMPTS,
    system_prompt="You are a careful assistant. Prefer clarity over verbosity.",
    agentic=True,
    context_factory=lambda prompt, index: {"runner": "notebook", "query_index": index, "prompt_length": len(prompt)},
    temperature=0.0,
    top_p=1.0,
    reasoning_effort="high",
    seed=42,
    max_tokens=1000,
    pause_seconds=1.0,
)


## Profile-Aware System Prompt Engineering

This section asks K2 to infer a concise, role-aware system prompt from a user profile and a concrete user query. The example below uses an associate financial-audit persona.


In [ ]:
AUDIT_ASSOCIATE_PROFILE = {
    "role": "Associate Financial Audit",
    "experience_years": 2,
    "industry_focus": "Technology and SaaS clients",
    "responsibilities": [
        "Perform walkthroughs and substantive testing",
        "Tie draft findings back to evidence",
        "Escalate control gaps and unsupported assertions",
    ],
    "strengths": [
        "Comfortable with audit workpapers and assertion language",
        "Can follow structured procedures and templates",
        "Needs help turning messy evidence into concise conclusions",
    ],
    "constraints": [
        "Limited time during fieldwork",
        "Needs outputs that are precise, defensible, and easy to paste into workpapers",
        "Should avoid overclaiming beyond the provided evidence",
    ],
    "preferred_output_style": [
        "Concise bullets",
        "Explicit linkage to assertions and evidence gaps",
        "Actionable next steps",
    ],
}

AUDIT_ASSOCIATE_QUERY = (
    "Review this draft revenue testing summary for a SaaS client and tell me where the support is weak, "
    "which audit assertions are most at risk, and what follow-up procedures I should perform next."
)

SYSTEM_PROMPT_ENGINEER_SYSTEM = (
    "You design high-signal system prompts for enterprise copilots. "
    "Generate a system prompt that fits the user profile and the user query. "
    "The system prompt must be precise, concise, practical, and safe for professional audit work. "
    "Return exactly one JSON object and nothing else. No analysis, no preamble, no markdown, and no XML tags."
)

def build_system_prompt_engineering_request(user_profile: dict, user_query: str) -> str:
    return (
        "Create a precise and concise system prompt for the assistant that will answer the user's query.\n"
        "Requirements:\n"
        "- Optimize for the user's role, likely skill level, and working context.\n"
        "- Make the assistant analytical, evidence-aware, and careful with unsupported claims.\n"
        "- Keep the system prompt short enough to be practical in production.\n"
        "- The system prompt should guide tone, structure, and risk handling.\n"
        "- Do not mention the hidden profile directly in the final system prompt.\n"
        "- Do not mention chain-of-thought.\n"
        "- Return exactly one JSON object with keys system_prompt and why_it_fits.\n"
        "- Output only the JSON object and nothing else.\n"
        "- Keep system_prompt under 110 words and why_it_fits under 2 sentences.\n\n"
        f"User profile:\n{json.dumps(user_profile, indent=2)}\n\n"
        f"User query:\n{user_query}"
    )


system_prompt_engineering_request = build_system_prompt_engineering_request(
    AUDIT_ASSOCIATE_PROFILE,
    AUDIT_ASSOCIATE_QUERY,
)

print("Audit associate profile and target query loaded.")
print(json.dumps({
    "role": AUDIT_ASSOCIATE_PROFILE["role"],
    "query": AUDIT_ASSOCIATE_QUERY,
}, indent=2))


In [ ]:
system_prompt_generation_result = call_k2(
    user_prompt=system_prompt_engineering_request,
    system_prompt=SYSTEM_PROMPT_ENGINEER_SYSTEM,
    context={"runner": "notebook", "mode": "profile_aware_system_prompt_generation"},
    agentic=True,
    temperature=0.0,
    top_p=1.0,
    reasoning_effort="low",
    seed=42,
    max_tokens=500,
)

show_result(system_prompt_generation_result)

generated_prompt_payload = parse_json_string(assistant_content(system_prompt_generation_result["raw_json"]))
GENERATED_AUDIT_SYSTEM_PROMPT = None
if isinstance(generated_prompt_payload, dict):
    GENERATED_AUDIT_SYSTEM_PROMPT = generated_prompt_payload.get("system_prompt")

print("\nGenerated system prompt:\n")
print(GENERATED_AUDIT_SYSTEM_PROMPT or "<no system prompt extracted>")


In [ ]:
if not GENERATED_AUDIT_SYSTEM_PROMPT:
    raise RuntimeError("No generated system prompt was extracted from the previous cell.")

engineered_prompt_response = call_k2(
    user_prompt=AUDIT_ASSOCIATE_QUERY,
    system_prompt=GENERATED_AUDIT_SYSTEM_PROMPT,
    context={"runner": "notebook", "mode": "profile_aware_answer_test"},
    agentic=True,
    temperature=0.0,
    top_p=1.0,
    seed=42,
    max_tokens=900,
)

show_result(engineered_prompt_response)


In [ ]:
output_dir = REPO_ROOT / "backend" / "data"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "k2_raw_query_results.json"

def serialize_result(item: dict | None) -> dict | None:
    if not isinstance(item, dict):
        return None
    return {
        "query_index": item.get("query_index"),
        "user_prompt": item.get("user_prompt"),
        "ok": item.get("ok"),
        "status_code": item.get("status_code"),
        "url": item.get("url"),
        "request_payload": item.get("request_payload"),
        "raw_json": item.get("raw_json"),
        "raw_text": item.get("raw_text"),
    }

payload_to_save = {
    "batch_results": [serialize_result(item) for item in batch_results],
    "profile_aware_generation": serialize_result(globals().get("system_prompt_generation_result")),
    "profile_aware_response": serialize_result(globals().get("engineered_prompt_response")),
    "generated_system_prompt": globals().get("GENERATED_AUDIT_SYSTEM_PROMPT"),
}

output_path.write_text(json.dumps(payload_to_save, indent=2), encoding="utf-8")
print(f"Saved raw results to: {output_path}")
